# Session 4 — Python for Market Risk
## Downloading Stock Data, Return Distributions & Portfolio Risk (VaR / CVaR)

**MSc Finance | Risk Modelling Course**

---

### Learning Objectives
By the end of this session you will be able to:
1. Download and clean historical price data using `yfinance`
2. Compute log-returns and understand their statistical properties
3. Visualise return distributions and rolling volatility with `seaborn`
4. Build a simple multi-asset portfolio and compute its risk metrics
5. Calculate **Historical VaR** and **CVaR** (Expected Shortfall) at multiple confidence levels
6. Appreciate how volatility regimes change over time (pre/post COVID case)

---

### Style note — method chaining
Throughout this notebook we favour **method chaining** (inspired by R's `dplyr` pipes `%>%`).  
In pandas this means stringing `.method()` calls together rather than creating lots of
intermediate variables.  The pattern looks like:

```python
result = (
    df
    .pipe(some_function)        # custom function injection
    .assign(new_col=...)        # add / transform columns
    .query("condition")         # filter rows
    .groupby("key")             # aggregate
    .agg({"col": "mean"})
    .rename(columns={"col": "nice_name"})
    .reset_index()
)
```

This keeps code readable and makes the data transformation story easy to follow.


## 0. Imports & Configuration

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

# ── Data & numerics ───────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import yfinance as yf

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# ── Seaborn global theme ──────────────────────────────────────────────────────
# 'whitegrid' gives clean axes; palette 'tab10' is colourblind-friendly
sns.set_theme(style="whitegrid", palette="tab10", font_scale=1.15)

# ── Display options ───────────────────────────────────────────────────────────
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 20)

print("All libraries loaded successfully.")


## 1. Downloading Historical Price Data

We pull **adjusted closing prices** for five assets that span different sectors,
giving us a nicely diversified illustrative portfolio:

| Ticker | Company / Index | Sector |
|--------|----------------|--------|
| AAPL   | Apple Inc.      | Technology |
| JPM    | JPMorgan Chase  | Financials |
| XOM    | ExxonMobil      | Energy |
| KO     | Coca-Cola       | Consumer Staples |
| ^GSPC  | S&P 500 Index   | Broad Market Benchmark |

We deliberately span **2018–2024** to capture normal markets, the COVID crash (Feb–Mar 2020),
and the post-pandemic inflation / rate-rise period.


In [ ]:
# ── Define our universe ───────────────────────────────────────────────────────
TICKERS    = ["AAPL", "JPM", "XOM", "KO", "^GSPC"]
START_DATE = "2018-01-01"
END_DATE   = "2024-12-31"

# ── Download via yfinance ──────────────────────────────────────────────────────
# auto_adjust=True gives dividend / split adjusted prices automatically
raw = yf.download(TICKERS, start=START_DATE, end=END_DATE,
                  auto_adjust=True, progress=False)

# yfinance returns a MultiIndex column like (field, ticker).
# We only need 'Close' prices — select and clean in one chain.
prices = (
    raw["Close"]                          # select the 'Close' level
    .rename_axis("Date")                  # make index name explicit
    .rename_axis("Ticker", axis="columns")# make column axis name explicit
    .dropna(how="all")                    # drop rows where ALL prices are NaN
    .ffill()                              # forward-fill any remaining gaps (holidays)
)

print(f"Price matrix shape: {prices.shape}  ({prices.shape[0]} trading days × {prices.shape[1]} assets)")
prices.tail()


## 2. Computing Log-Returns

We use **log-returns** rather than simple returns because:
- They are time-additive: a multi-period log-return is the sum of single-period ones
- They are approximately normally distributed for short horizons
- They eliminate the lower bound at −100% that afflicts simple returns

$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$$


In [ ]:
# ── Log-return computation via method chain ───────────────────────────────────
# np.log(prices / prices.shift(1)) is equivalent to prices.pct_change() on log scale
returns = (
    np.log(prices / prices.shift(1))     # compute daily log-returns
    .dropna()                             # drop the first row (NaN from shift)
)

print(f"Returns matrix shape: {returns.shape}")
returns.describe().T


## 3. Summary Statistics — Annualised Risk & Return

We annualise by multiplying mean returns by **252** (trading days per year)
and standard deviation by **√252**.

Note: The S&P 500 (`^GSPC`) serves as our benchmark throughout.


In [ ]:
TRADING_DAYS = 252

def annualised_stats(returns_df):
    """
    Given a DataFrame of daily log-returns, return a summary table
    with annualised mean return, volatility, Sharpe-like ratio, skew and kurtosis.
    Risk-free rate assumed 0 for simplicity (adjust RF below if needed).
    """
    RF = 0.04 / TRADING_DAYS  # ~4% annualised risk-free rate, daily

    stats = (
        returns_df
        .agg(["mean", "std", "skew", "kurt"])     # four moments
        .T                                          # transpose so assets are rows
        .rename(columns={"mean": "Daily Mean",
                         "std":  "Daily Std",
                         "skew": "Skewness",
                         "kurt": "Excess Kurtosis"})
        .assign(
            Ann_Return = lambda df: df["Daily Mean"] * TRADING_DAYS,
            Ann_Vol    = lambda df: df["Daily Std"]  * np.sqrt(TRADING_DAYS),
        )
        .assign(
            Sharpe = lambda df: (df["Ann_Return"] - 0.04) / df["Ann_Vol"]
        )
        [["Ann_Return", "Ann_Vol", "Sharpe", "Skewness", "Excess Kurtosis"]]
    )
    return stats

stats = annualised_stats(returns)
stats


## 4. Visualising Return Distributions

A normal distribution assumption is baked into many risk models (including Black-Scholes).
Let's see how realistic that is — financial returns are typically **fat-tailed** (leptokurtic)
and often slightly **negatively skewed** (large losses are more common than a normal predicts).


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, ticker in enumerate(TICKERS):
    ax = axes[i]
    data = returns[ticker].dropna()

    # ── Seaborn histplot with KDE overlay ─────────────────────────────────────
    sns.histplot(data, bins=80, kde=True, stat="density",
                 color=sns.color_palette("tab10")[i], alpha=0.55, ax=ax)

    # ── Overlay a fitted normal for comparison ─────────────────────────────────
    from scipy.stats import norm
    mu, sigma = data.mean(), data.std()
    x = np.linspace(data.min(), data.max(), 300)
    ax.plot(x, norm.pdf(x, mu, sigma), "k--", lw=1.5, label="Normal fit")

    # ── Annotation ────────────────────────────────────────────────────────────
    skew_val = data.skew()
    kurt_val = data.kurt()
    ax.set_title(f"{ticker}", fontsize=13, fontweight="bold")
    ax.set_xlabel("Daily Log-Return")
    ax.text(0.03, 0.95, f"Skew={skew_val:.2f}\nKurt={kurt_val:.2f}",
            transform=ax.transAxes, va="top", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))
    ax.legend(fontsize=8)

# ── Remove the unused 6th subplot ─────────────────────────────────────────────
axes[5].set_visible(False)

fig.suptitle("Daily Log-Return Distributions vs Normal Fit\n(2018–2024)",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print("\nKey insight: The KDE (solid) has HIGHER peaks and FATTER tails than the normal (dashed).")
print("This is 'excess kurtosis' — large moves happen more often than a normal distribution predicts.")


## 5. Rolling Volatility — Spotting Regime Changes

**Static** volatility (one number for the whole period) ignores the fact that markets
are calm for extended stretches and then violently volatile for short bursts.
Rolling volatility makes this visible.

We use a **21-day** rolling window (≈ 1 trading month) and annualise.


In [ ]:
ROLL_WINDOW = 21  # trading days ≈ 1 month

rolling_vol = (
    returns
    .rolling(window=ROLL_WINDOW)          # rolling window object
    .std()                                 # rolling std dev
    .mul(np.sqrt(TRADING_DAYS))           # annualise
    .dropna()
    .rename_axis("Date")
)

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

for ticker in TICKERS:
    ax.plot(rolling_vol.index, rolling_vol[ticker], lw=1.3, label=ticker)

# ── Annotate key events ────────────────────────────────────────────────────────
events = {
    "COVID crash\n(Mar 2020)":  "2020-03-16",
    "Fed hikes begin\n(Mar 2022)": "2022-03-16",
}
for label, date in events.items():
    ax.axvline(pd.Timestamp(date), color="red", lw=1.2, ls="--", alpha=0.7)
    ax.text(pd.Timestamp(date), ax.get_ylim()[1] * 0.92, label,
            ha="center", fontsize=8.5, color="red",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.6))

ax.set_title(f"{ROLL_WINDOW}-Day Rolling Annualised Volatility", fontsize=14, fontweight="bold")
ax.set_ylabel("Annualised Volatility")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

print("\nKey insight: Volatility CLUSTERS — it is persistently high or low, not random.")
print("The COVID spike in March 2020 is clearly visible across all assets.")


## 6. Correlation Matrix

Correlation is the engine of diversification.
A portfolio's risk is **not** the average of individual risks — correlations below 1
reduce total portfolio volatility.

We compute correlations in two regimes to show how they shift under stress.


In [ ]:
# ── Define two market regimes ─────────────────────────────────────────────────
regimes = {
    "Normal (2018–2019)": ("2018-01-01", "2019-12-31"),
    "Stress (COVID 2020)": ("2020-01-01", "2020-12-31"),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (label, (start, end)) in zip(axes, regimes.items()):
    corr = (
        returns
        .loc[start:end]          # filter to regime dates
        .corr()                  # pairwise Pearson correlation
    )
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # upper triangle mask

    sns.heatmap(
        corr, ax=ax,
        annot=True, fmt=".2f",
        cmap="RdYlGn", center=0, vmin=-1, vmax=1,
        mask=mask,                # show lower triangle only
        linewidths=0.5,
        cbar_kws={"shrink": 0.8},
    )
    ax.set_title(f"Correlation Matrix\n{label}", fontsize=12, fontweight="bold")

plt.suptitle("Correlations Rise During Stress — Diversification Fails When You Need It Most",
             fontsize=12, style="italic", y=1.02)
plt.tight_layout()
plt.show()


## 7. Building a Simple Equal-Weight Portfolio

We allocate capital equally across the four stocks (excluding the index benchmark).
This is the simplest possible diversification strategy — and often surprisingly hard to beat.

Portfolio daily return: $r_p = \sum_i w_i \cdot r_i$ where $w_i = 0.25$ for each stock.


In [ ]:
# ── Define portfolio (equal-weight, 4 stocks, no index) ───────────────────────
PORTFOLIO_TICKERS = ["AAPL", "JPM", "XOM", "KO"]
N_ASSETS = len(PORTFOLIO_TICKERS)
weights  = np.array([1 / N_ASSETS] * N_ASSETS)  # equal weights: 0.25 each

port_returns = (
    returns[PORTFOLIO_TICKERS]                # select only portfolio assets
    .assign(Portfolio=lambda df: df.dot(weights))  # weighted sum → portfolio return
)

print("Daily portfolio return statistics:")
print(port_returns["Portfolio"].describe())
print(f"\nAnnualised Return : {port_returns['Portfolio'].mean() * TRADING_DAYS:.2%}")
print(f"Annualised Volatility: {port_returns['Portfolio'].std() * np.sqrt(TRADING_DAYS):.2%}")


## 8. Historical Value at Risk (VaR) and CVaR (Expected Shortfall)

### What is VaR?
**Value at Risk** at confidence level $\alpha$ answers:
> *"What is the worst daily loss I can expect to suffer on $X\%$ of trading days?"*

Formally: $\text{VaR}_{\alpha} = -\text{quantile}_{1-\alpha}(r)$

At 95% confidence: on 95% of days, losses will NOT exceed VaR₉₅.
Equivalently, VaR is exceeded on 5% of days — roughly 13 trading days per year.

### What is CVaR / Expected Shortfall?
CVaR fixes VaR's main weakness: VaR says nothing about *how bad* the bad days are.

**CVaR** = average loss **given** that we are already beyond the VaR threshold:
$$\text{CVaR}_{\alpha} = -\mathbb{E}[r \mid r < -\text{VaR}_{\alpha}]$$

CVaR is **coherent** (sub-additive), making it preferred by regulators (Basel III uses CVaR).


In [ ]:
def compute_var_cvar(returns_series, confidence_levels=(0.90, 0.95, 0.99)):
    """
    Compute Historical (non-parametric) VaR and CVaR for a returns series.

    Parameters
    ----------
    returns_series   : pd.Series of daily log-returns
    confidence_levels: tuple of floats, e.g. (0.95, 0.99)

    Returns
    -------
    pd.DataFrame with VaR and CVaR at each confidence level (as positive loss figures)
    """
    rows = []
    for cl in confidence_levels:
        var   = -returns_series.quantile(1 - cl)           # negate → positive loss
        cvar  = -returns_series[returns_series < -var].mean()  # average of tail losses
        rows.append({"Confidence": f"{cl:.0%}", "VaR": var, "CVaR": cvar})

    return pd.DataFrame(rows).set_index("Confidence")


# ── Compute for each asset + portfolio ────────────────────────────────────────
all_series = port_returns.copy()  # includes individual stocks + portfolio column

results = {}
for col in all_series.columns:
    results[col] = compute_var_cvar(all_series[col])

# ── Display for the portfolio ──────────────────────────────────────────────────
print("=== Portfolio VaR & CVaR ===")
print(results["Portfolio"])
print()

# ── Cross-asset comparison at 95% ─────────────────────────────────────────────
comparison_95 = (
    pd.DataFrame({
        ticker: results[ticker].loc["95%"]
        for ticker in all_series.columns
    })
    .T
    .assign(
        Ann_VaR  = lambda df: df["VaR"]  * np.sqrt(TRADING_DAYS),
        Ann_CVaR = lambda df: df["CVaR"] * np.sqrt(TRADING_DAYS),
    )
    .rename(columns={"VaR": "Daily VaR", "CVaR": "Daily CVaR"})
)

print("=== Cross-Asset Comparison at 95% Confidence ===")
comparison_95


In [ ]:
# ── Visualise VaR on the return distribution ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, ticker in zip(axes, ["^GSPC", "Portfolio"]):
    data   = all_series[ticker].dropna()
    var_95 = results[ticker].loc["95%", "VaR"]
    var_99 = results[ticker].loc["99%", "VaR"]

    # Full distribution
    sns.histplot(data, bins=80, kde=True, stat="density",
                 color="steelblue", alpha=0.4, ax=ax, label="Return dist.")

    # Shade the tail losses
    tail_data = data[data < -var_99]
    ax.fill_between(
        sorted(tail_data), 0,
        [sns.kdeplot(data, ax=None).get_lines()[0].get_ydata().max() * 0.1] * len(tail_data),
        color="darkred", alpha=0.5, label="CVaR region (99%)"
    )

    # VaR vertical lines
    ax.axvline(-var_95, color="orange", lw=2, ls="--", label=f"VaR 95% = {var_95:.2%}")
    ax.axvline(-var_99, color="red",    lw=2, ls="--", label=f"VaR 99% = {var_99:.2%}")

    ax.set_title(f"{ticker} — Return Distribution with VaR", fontweight="bold")
    ax.set_xlabel("Daily Log-Return")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


## 9. Regime Analysis — Pre vs Post COVID Volatility

This is a concrete example of why **static volatility assumptions are dangerous**.
Models calibrated on 2018–2019 data would dramatically underestimate risk during 2020.


In [ ]:
# ── Define regimes ────────────────────────────────────────────────────────────
regime_map = {
    "Pre-COVID (2018–2019)": ("2018-01-01", "2019-12-31"),
    "COVID Crash (2020)":    ("2020-01-01", "2020-12-31"),
    "Recovery (2021–2022)":  ("2021-01-01", "2022-12-31"),
}

regime_stats = (
    pd.concat(
        {
            label: annualised_stats(returns[PORTFOLIO_TICKERS].loc[start:end])
            for label, (start, end) in regime_map.items()
        },
        names=["Regime", "Ticker"]
    )
    .reset_index()
)

# ── Plot annualised volatility by regime ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))

sns.barplot(
    data=regime_stats,
    x="Ticker", y="Ann_Vol",
    hue="Regime",
    palette=["#4CAF50", "#F44336", "#2196F3"],
    ax=ax
)

ax.set_title("Annualised Volatility by Market Regime", fontsize=14, fontweight="bold")
ax.set_ylabel("Annualised Volatility")
ax.set_xlabel("")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend(title="Regime", loc="upper left")
plt.tight_layout()
plt.show()

print("\nKey insight: COVID-era volatility was 2–4× the pre-COVID baseline.")
print("A risk model using pre-COVID vol would have massively under-reserved capital.")


## 10. Cumulative Returns — Growth of £1 Invested

Let's close with a practical view: what would £1 invested at the start of 2018 be worth by end 2024?


In [ ]:
# ── Cumulative log-return → price index ───────────────────────────────────────
# exp(cumulative log-returns) restores the compound growth factor
cum_returns = (
    port_returns
    .cumsum()                   # cumulative sum of log-returns
    .apply(np.exp)              # convert log-cumulative → simple growth factor
    .rename(columns={"Portfolio": "Portfolio (EW)"})
)

fig, ax = plt.subplots(figsize=(13, 5))

for col in cum_returns.columns:
    lw = 2.5 if col == "Portfolio (EW)" else 1.2
    ls = "-"  if col == "Portfolio (EW)" else "--"
    ax.plot(cum_returns.index, cum_returns[col], lw=lw, ls=ls, label=col)

ax.axhline(1, color="grey", lw=0.8, ls=":")
ax.set_title("Cumulative Growth of £1 Invested (2018–2024)", fontsize=14, fontweight="bold")
ax.set_ylabel("Portfolio Value (£ per £1 invested)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend()
plt.tight_layout()
plt.show()


## Summary & Key Takeaways

| Concept | What we learned |
|---------|----------------|
| Log-returns | Time-additive, approximately normal but with fat tails in practice |
| Rolling volatility | Volatility clusters — it is NOT constant (this will matter for Black-Scholes) |
| Correlations | Rise dramatically during stress — diversification weakens precisely when needed |
| Historical VaR | Simple but relies entirely on the past; says nothing about tail shape |
| CVaR | Tells us the *average* loss in the tail — preferred by regulators (Basel III) |
| Regime analysis | Static models calibrated on calm periods dramatically understate crisis risk |

### Looking ahead to Session 5
We'll take the **GBM / lognormal** model of stock prices seriously and use it to price options
analytically (Black-Scholes formula) and by simulation — and then examine where it breaks down.
